# Multi-Agent Hierarchical Documentation Assistant

This notebook demonstrates the complete end-to-end pipeline for generating project documentation.

## Features
- **Phase 1**: Static code analysis using tree-sitter (multi-language support)
- **Phase 2**: Google-style docstring generation
- **Phase 3**: README generation (6 sections)
- **Phase 4**: Validation and iterative improvement
- **Phase 5**: Final quality evaluation

## Memory Optimization
- 4-bit quantization for GPU compatibility
- Small model (1.5B params)
- Content-based caching

## Installation

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torch sentencepiece bitsandbytes tree-sitter tree-sitter-languages networkx psutil pydantic

## Clone Repository (if needed)

In [ ]:
# Clone this repository if running on Colab
import os

if not os.path.exists('Multi-agent_Hierarchical_Documentation'):
    !git clone https://github.com/SalmaHisham/Multi-agent_Hierarchical_Documentation.git
    os.chdir('Multi-agent_Hierarchical_Documentation')
else:
    os.chdir('Multi-agent_Hierarchical_Documentation')

## Setup Environment

In [ ]:
import os
import sys

# Reduce HF/Transformers noise
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Add current directory to path
if "." not in sys.path:
    sys.path.insert(0, ".")

## Import Modules

In [ ]:
from orchestrator import Orchestrator
from pathlib import Path
import json

## Configure Project Path

Set the path to the project you want to document.

In [ ]:
# Example: document this repository itself
PROJECT_PATH = "."

# Or specify a different project
# PROJECT_PATH = "/path/to/your/project"

## Initialize Orchestrator

The orchestrator coordinates all 5 phases of the pipeline.

In [ ]:
orchestrator = Orchestrator(
    repo_path=PROJECT_PATH,
    artifacts_dir="./artifacts",
    model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",  # 1.5B for speed
    device="auto",  # Use GPU if available
    quantize=True,  # 4-bit quantization for GPU
    use_structural_agent=True  # Use enhanced Phase 1
)

print(f"Project: {orchestrator.project_name}")
print(f"Path: {orchestrator.repo_path}")

## Phase 1: Static Analysis

Parse source code using tree-sitter to extract AST, dependencies, and components.
Supports Python, Java, JavaScript, TypeScript, C, C++, C#.

In [ ]:
phase1_results = orchestrator.run_phase1()

print(f"\nModules: {phase1_results['stats']['modules']}")
print(f"Functions: {phase1_results['stats']['functions']}")
print(f"Classes: {phase1_results['stats']['classes']}")

# Output locations
print(f"\nOutputs saved to:")
for key, path in phase1_results.get('artifacts', {}).items():
    print(f"  • {key}: {path}")

## Phase 2: Docstring Generation

Generate Google-style docstrings using lightweight LLM with caching.

In [ ]:
phase2_results = orchestrator.run_phase2()

print(f"\nTotal: {phase2_results['stats']['total']}")
print(f"Cached: {phase2_results['stats']['cached']}")
print(f"Generated: {phase2_results['stats']['generated']}")
print(f"\nOutput: {phase2_results['output_path']}")

## Phase 3: README Generation

Generate comprehensive README with 6 required sections and working badges.

In [ ]:
phase3_results = orchestrator.run_phase3()

print(f"\nREADME generated: {phase3_results['readme_path']}")

## Phase 4: Validation

Validate README sections using heuristics.

In [ ]:
phase4_results = orchestrator.run_phase4()

print(f"\nAll valid: {phase4_results['all_valid']}")
for section, result in phase4_results['sections'].items():
    status = "✅" if result['valid'] else "⚠️"
    print(f"{status} {section}")

## Phase 5: Evaluation

Evaluate documentation quality on 4 metrics.

In [ ]:
phase5_results = orchestrator.run_phase5()

evaluation = phase5_results['evaluation']
print(f"\nOverall Score: {evaluation['overall']:.1f}/10")
print(f"Clarity: {evaluation.get('clarity', 0)}")
print(f"Completeness: {evaluation.get('completeness', 0)}")
print(f"Consistency: {evaluation.get('consistency', 0)}")
print(f"Usability: {evaluation.get('usability', 0)}")
print(f"\nReport: {phase5_results['report_path']}")

## View Generated README

In [ ]:
readme_path = Path(phase3_results['readme_path'])
if readme_path.exists():
    readme_content = readme_path.read_text()
    print(readme_content)
else:
    print("README not found")

## View All Output Locations

In [ ]:
print("📂 All Output Locations:")
print(f"\nPhase 1 (Analysis):")
for key, path in phase1_results.get('artifacts', {}).items():
    print(f"  • {key}: {path}")
print(f"\nPhase 2 (Docstrings):")
print(f"  • docstrings: {phase2_results['output_path']}")
print(f"  • cache: {orchestrator.artifacts_dir}/cache")
print(f"\nPhase 3 (README):")
print(f"  • README: {phase3_results['readme_path']}")
print(f"\nPhase 4 (Validation):")
print(f"  • (in-memory only)")
print(f"\nPhase 5 (Evaluation):")
print(f"  • report: {phase5_results['report_path']}")

## Cleanup

In [ ]:
# Free GPU memory
orchestrator.cleanup()
print("Cleanup complete!")

## Running All Phases at Once

Alternatively, you can run all phases with a single command:

In [ ]:
# Create a fresh orchestrator
orchestrator = Orchestrator(
    repo_path=PROJECT_PATH,
    artifacts_dir="./artifacts",
    model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    device="auto",
    quantize=True,
    use_structural_agent=True,
)

# Run all phases
all_results = orchestrator.run_all()

# Cleanup
orchestrator.cleanup()

DeepWiki Builder + RAG QA (Qwen-only) + Optional LoRA Fine-tuning

In [ ]:
"""
DeepWiki Builder + RAG QA (Qwen-only) + Optional LoRA Fine-tuning
(CORRECTED VERSION: cleaner outputs + eval + training stub generation)

What’s improved:
- Deterministic generation (do_sample=False, eos + pad token ids)
- Strict output contract + output sanitization (prevents "Human:"/"Assistant:" leakage)
- Helper to build the exact prompt used (for LoRA training)
- Evaluation runner to compare base vs LoRA on a fixed question set
- Training-stub generator (train_stubs.jsonl) to fill with gold answers
"""

from __future__ import annotations

import os
import json
import re
import glob
import sys
import base64
import time
import subprocess
from dataclasses import dataclass
from collections import Counter
from typing import Any, Dict, List, Tuple, Optional

import requests


# ------------------------
# Mermaid (no npm)
# ------------------------
def render_mermaid_png(
    mermaid_code: str,
    out_png: str,
    scale: int = 3,
    timeout: int = 30
) -> Optional[str]:
    try:
        encoded = base64.urlsafe_b64encode(mermaid_code.encode("utf-8")).decode("ascii")
        url = f"https://mermaid.ink/img/{encoded}?scale={scale}"
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        with open(out_png, "wb") as f:
            f.write(r.content)
        return out_png
    except Exception as e:
        print("⚠️ Mermaid PNG export failed:", e)
        return None


# ------------------------
# Helpers
# ------------------------
def safe_id(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", s)


def deps_to_mermaid(deps: dict) -> str:
    lines = ["graph TD"]
    for e in deps["data"]["edges"]:
        src = safe_id(e["from"])
        dst = safe_id(e["to"])
        label = e.get("kind", "")
        if label:
            lines.append(f"  {src} -->|{label}| {dst}")
        else:
            lines.append(f"  {src} --> {dst}")
    return "\n".join(lines)


def chunk_text(text: str, size: int = 900, overlap: int = 150) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    out, i = [], 0
    while i < len(text):
        out.append(text[i : i + size])
        i += max(1, size - overlap)
    return out


def load_lines(path: str) -> List[str]:
    return [ln.strip() for ln in open(path, "r", encoding="utf-8") if ln.strip()]


def write_jsonl(path: str, rows: List[dict]):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def read_jsonl(path: str) -> List[dict]:
    return [json.loads(x) for x in open(path, "r", encoding="utf-8") if x.strip()]


# ------------------------
# Build DeepWiki + RAG
# ------------------------
def build_deepwiki_from_artifacts(artifacts_dir: str) -> dict:
    try:
        import faiss
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        import faiss

    from sentence_transformers import SentenceTransformer

    repo_candidates = []
    for d in os.listdir(artifacts_dir):
        if d.startswith("."):
            continue
        full = os.path.join(artifacts_dir, d)
        if os.path.isdir(full) and os.path.exists(os.path.join(full, "dependencies_normalized.json")):
            repo_candidates.append(d)

    if not repo_candidates:
        raise FileNotFoundError("No repo with dependencies_normalized.json found")

    repo_name = sorted(repo_candidates)[0]
    dep_path = os.path.join(artifacts_dir, repo_name, "dependencies_normalized.json")

    with open(dep_path, "r", encoding="utf-8") as f:
        deps = json.load(f)

    nodes = deps["data"]["nodes"]
    edges = deps["data"]["edges"]

    mermaid_graph = deps_to_mermaid(deps)
    mmd_path = os.path.join(artifacts_dir, "dataflow.mmd")
    with open(mmd_path, "w", encoding="utf-8") as f:
        f.write(mermaid_graph)

    png_path = render_mermaid_png(mermaid_graph, os.path.join(artifacts_dir, "dataflow.png"))

    WIKI_DIR = os.path.join(artifacts_dir, repo_name, "wiki")
    os.makedirs(WIKI_DIR, exist_ok=True)

    def save_md(name: str, text: str):
        with open(os.path.join(WIKI_DIR, name), "w", encoding="utf-8") as f:
            f.write(text.strip() + "\n")

    in_deg, out_deg = Counter(), Counter()
    for e in edges:
        out_deg[e["from"]] += 1
        in_deg[e["to"]] += 1

    internal_nodes = [n for n in nodes if not str(n).startswith("external:")]
    external_nodes = [n for n in nodes if str(n).startswith("external:")]

    top_internal = sorted(internal_nodes, key=lambda n: in_deg[n] + out_deg[n], reverse=True)[:12]

    save_md("01_System_Overview.md", f"""
# System Overview
- Nodes: {len(nodes)}
- Edges: {len(edges)}
- Internal: {len(internal_nodes)}
- External: {len(external_nodes)}
""")

    save_md("02_Dataflow.md", f"""
```mermaid
{mermaid_graph}
```
""")

    save_md("03_Key_Components.md", "# Key Components\n\n" +
            "\n".join(f"- {n} (deg={in_deg[n] + out_deg[n]})" for n in top_internal))

    save_md("04_External_Dependencies.md", "# External Dependencies\n\n" +
            "\n".join(f"- {n}" for n in external_nodes))

    save_md("README.md", "# DeepWiki Index\n")

    wiki_files = sorted(glob.glob(os.path.join(WIKI_DIR, "*.md")))

    docs, metas = [], []
    for p in wiki_files:
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read()
        for i, ch in enumerate(chunk_text(txt)):
            docs.append(ch)
            metas.append({"file": os.path.basename(p), "chunk": i})

    embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
    X = embedder.encode(docs, normalize_embeddings=True).astype("float32")

    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)

    def retrieve(q: str, k: int = 4):
        qv = embedder.encode([q], normalize_embeddings=True).astype("float32")
        D, I = index.search(qv, k)
        return [(float(D[0][i]), metas[j], docs[j]) for i, j in enumerate(I[0])]

    return {
        "repo": repo_name,
        "wiki_dir": WIKI_DIR,
        "mermaid_mmd": mmd_path,
        "mermaid_png": png_path,
        "retrieve": retrieve,
        "docs": docs,
        "metas": metas,
        "embedder": embedder,
        "faiss_index": index,
    }


# ------------------------
# Qwen-only QA + LoRA
# ------------------------
@dataclass
class QwenSpec:
    name: str
    model_id: str = "Qwen/Qwen2.5-1.5B-Instruct"
    adapter_dir: Optional[str] = None


class DeepWikiQA:
    def __init__(self, bundle: dict):
        self.bundle = bundle
        self.retrieve = bundle["retrieve"]
        self._loaded: Dict[str, Tuple[Any, Any]] = {}

    def _load(self, spec: QwenSpec):
        if spec.name in self._loaded:
            return self._loaded[spec.name]

        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch

        tok = AutoTokenizer.from_pretrained(spec.model_id, use_fast=True)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            spec.model_id,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )

        if spec.adapter_dir:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, spec.adapter_dir)

        self._loaded[spec.name] = (tok, model)
        return tok, model

    def build_prompt(self, question: str, k: int = 4, max_chars_per_hit: int = 800) -> Tuple[str, str]:
        """
        Returns (prompt, context) exactly as used for generation.
        """
        hits = self.retrieve(question, k)

        context = "\n".join(
            f"- ({m['file']}#{m['chunk']}) {t[:max_chars_per_hit].replace(chr(10), ' ')}"
            for _, m, t in hits
        )

        prompt = f"""You are a technical documentation assistant.

Task: Answer the question using ONLY the context.

STRICT OUTPUT RULES:
- Output MUST be either:
  (A) a bullet list (3–9 bullets), OR
  (B) exactly: Not found in documentation.
- Output NOTHING else.
- Do NOT repeat the context.
- Do NOT include headings, explanations, or code blocks.
- Do NOT include role labels like "Human:" or "Assistant:".

Context:
{context}

Question:
{question}

Answer:
"""
        return prompt, context

    def _sanitize(self, out: str) -> str:
        out = out.strip()

        # Enforce fallback-only
        if "Not found in documentation." in out:
            return "Not found in documentation."

        # Remove role leakage if present
        for bad in ("Human:", "Assistant:"):
            out = out.replace(bad, "").strip()

        # If it contains headings/code blocks, output is untrusted; return fallback
        bad_markers = ("```", "System Overview", "Dataflow", "Key Components")
        if any(m in out for m in bad_markers):
            return "Not found in documentation."

        # Trim length (avoid rambling)
        lines = [ln.strip() for ln in out.splitlines() if ln.strip()]
        out = "\n".join(lines[:12]).strip()

        return out

    def ask(self, spec: QwenSpec, question: str, k: int = 4, max_new_tokens: int = 250) -> str:
        tok, model = self._load(spec)
        prompt, _ = self.build_prompt(question, k=k)

        x = tok(prompt, return_tensors="pt").to(model.device)
        y = model.generate(
            **x,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.eos_token_id,
        )
        text = tok.decode(y[0], skip_special_tokens=True)

        out = text.split("Answer:", 1)[-1].strip() if "Answer:" in text else text.strip()
        return self._sanitize(out)

    # ------------------------
    # Evaluation + Training stubs
    # ------------------------
    def evaluate(self, spec: QwenSpec, questions: List[str], out_jsonl: str, k: int = 4):
        rows = []
        for q in questions:
            ans = self.ask(spec, q, k=k)
            rows.append({"model": spec.name, "question": q, "answer": ans, "ts": time.time()})
        write_jsonl(out_jsonl, rows)
        print(f"✅ Wrote: {out_jsonl}")

    def make_training_stubs(self, questions: List[str], out_jsonl: str = "train_stubs.jsonl", k: int = 4):
        """
        Writes prompt stubs for LoRA training. You fill 'response' with the ideal answer.
        """
        rows = []
        for q in questions:
            prompt, _ = self.build_prompt(q, k=k)
            rows.append({"prompt": prompt, "response": ""})
        write_jsonl(out_jsonl, rows)
        print(f"✅ Wrote: {out_jsonl} (fill responses, then rename to train.jsonl)")


# ------------------------
# Usage example
# ------------------------
"""
# 1) Build bundle
bundle = build_deepwiki_from_artifacts(artifacts_dir)
qa = DeepWikiQA(bundle)

# 2) Fixed question set (file or list)
questions = load_lines("eval_questions.txt")

# 3) Evaluate base
base = QwenSpec(name="qwen-base")
qa.evaluate(base, questions, out_jsonl="eval_base.jsonl")

# 4) Create training stubs from bad questions
qa.make_training_stubs(questions[:30], out_jsonl="train_stubs.jsonl")

# 5) After you fine-tune and produce adapter dir:
lora = QwenSpec(name="qwen-lora", adapter_dir="qwen_lora_adapter")
qa.evaluate(lora, questions, out_jsonl="eval_lora.jsonl")
"""


In [ ]:
artifacts_dir = "./artifacts"   # change this

bundle = build_deepwiki_from_artifacts(artifacts_dir)

In [ ]:

# 2) Create the QA wrapper
qa = DeepWikiQA(bundle)

# 3) Choose model (base Qwen)
spec = QwenSpec(name="qwen-base")  # or QwenSpec(name="qwen-lora", adapter_dir="qwen_lora_adapter")

# 4) Ask one question
question = "How many nodes and edges are in the system?"
answer = qa.ask(spec, question, k=4)

print(answer)

In [ ]:
def finetune_lora_sft_auto(
    base_model_id: str,
    train_jsonl_path: str,
    output_adapter_dir: str,
    max_steps: int = 200,
    lr: float = 2e-4,
    batch_size: int = 1,
    grad_accum: int = 8,
    max_seq_len: int = 1024,
    force_fp16_fallback: bool = False,
):
    """
    Tries QLoRA (4-bit) first (needs bitsandbytes).
    If bitsandbytes is missing/old/incompatible OR QLoRA fails at runtime,
    automatically falls back to FP16 LoRA with memory-saving settings.

    Output: saves LoRA adapter to output_adapter_dir
    """
    import sys
    import subprocess

    def _pip_install(pkgs):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs)

    # -------------------------
    # Ensure core deps exist
    # -------------------------
    try:
        import datasets  # noqa
        import accelerate  # noqa
        import peft  # noqa
        import transformers  # noqa
    except Exception:
        _pip_install(["datasets", "accelerate", "peft", "transformers"])

    # -------------------------
    # bitsandbytes (optional for QLoRA)
    # -------------------------
    bnb_ok = False
    bnb_reason = ""
    if not force_fp16_fallback:
        try:
            import bitsandbytes  # noqa
            bnb_ok = True
        except Exception:
            # Try install/upgrade
            try:
                _pip_install(["bitsandbytes"])
                import bitsandbytes  # noqa
                bnb_ok = True
            except Exception as e:
                bnb_ok = False
                bnb_reason = f"bitsandbytes install/import failed: {e}"

        # If bnb is present, still try upgrading because 4-bit needs "latest"
        if bnb_ok:
            try:
                _pip_install(["bitsandbytes"])
            except Exception as e:
                # upgrade failure shouldn't stop; we'll attempt QLoRA anyway and fallback if it fails
                bnb_reason = f"bitsandbytes upgrade failed: {e}"

    # -------------------------
    # Common imports
    # -------------------------
    import torch
    from datasets import load_dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        TrainingArguments,
        Trainer,
        DataCollatorForLanguageModeling,
    )
    from peft import LoraConfig, get_peft_model

    # -------------------------
    # Load dataset
    # -------------------------
    ds = load_dataset("json", data_files=train_jsonl_path, split="train")

    # -------------------------
    # Tokenizer
    # -------------------------
    tok = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # -------------------------
    # Attempt QLoRA (4-bit) with safe fallback
    # -------------------------
    use_qlora = False
    model = None
    optim_name = "adamw_torch"
    fp16_flag = torch.cuda.is_available()

    if bnb_ok and not force_fp16_fallback:
        try:
            from transformers import BitsAndBytesConfig
            from peft import prepare_model_for_kbit_training

            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )

            print("✅ Using QLoRA (4-bit)")

            model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                device_map="auto",
                quantization_config=bnb_cfg,
            )

            # memory savers
            model.gradient_checkpointing_enable()
            model = prepare_model_for_kbit_training(model)

            optim_name = "paged_adamw_8bit"
            fp16_flag = True
            use_qlora = True

        except Exception as e:
            # This catches: "requires latest bitsandbytes", CUDA mismatch, quantizer errors, etc.
            print("⚠️ QLoRA failed; falling back to FP16 LoRA.")
            if bnb_reason:
                print("   Note:", bnb_reason)
            print("   Reason:", repr(e))
            use_qlora = False

    if not use_qlora:
        print("✅ Using FP16 LoRA fallback (no/failed QLoRA)")
        # Reduce memory footprint aggressively
        max_seq_len = min(max_seq_len, 512)

        model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )

        model.gradient_checkpointing_enable()
        optim_name = "adamw_torch"
        fp16_flag = torch.cuda.is_available()

    # -------------------------
    # LoRA config
    # -------------------------
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # -------------------------
    # Tokenize
    # -------------------------
    def tokenize(ex):
        text = ex["prompt"] + ex["response"]
        out = tok(
            text,
            truncation=True,
            max_length=max_seq_len,
            padding="max_length",
        )
        out["labels"] = out["input_ids"].copy()
        return out

    ds_tok = ds.map(tokenize, remove_columns=ds.column_names)

    collator = DataCollatorForLanguageModeling(tok, mlm=False)

    # -------------------------
    # Training args
    # -------------------------
    args = TrainingArguments(
        output_dir=output_adapter_dir,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        max_steps=max_steps,
        logging_steps=20,
        save_steps=100,
        save_total_limit=2,
        fp16=fp16_flag,
        report_to="none",
        remove_unused_columns=False,
        optim=optim_name,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok,
        data_collator=collator,
    )

    trainer.train()

    model.save_pretrained(output_adapter_dir)
    tok.save_pretrained(output_adapter_dir)
    print("✅ LoRA adapter saved to:", output_adapter_dir)


In [ ]:
finetune_lora_sft_auto(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    train_jsonl_path="./train_fixed.jsonl",
    output_adapter_dir="qwen_lora_adapter",
    max_steps=100,
    batch_size=1,
    grad_accum=8,
    max_seq_len=1024,
)

In [ ]:
lora = QwenSpec(name="qwen-lora", adapter_dir="qwen_lora_adapter")

# force reload by using a different name to avoid cache issues
lora2 = QwenSpec(name="qwen-lora-v2", adapter_dir="qwen_lora_adapter")

print(qa.ask(lora2, "What are the key components of the system?"))